## 기본 설정

In [1]:
# 필요한 라이브러리 설치
!apt-get -qq update && apt-get -qq install -y ffmpeg sox libsox-dev
!pip -q install librosa soundfile huggingface_hub pandas matplotlib scikit-learn onnxruntime-gpu
!pip -q install git+https://github.com/speechbrain/speechbrain.git@develop

!test -d /content/CosyVoice || git clone --recursive -q https://github.com/QwenAudio/CosyVoice.git /content/CosyVoice
!pip -q install "transformers==4.51.3" "diffusers==0.29.0" "conformer==0.3.2" HyperPyYAML hydra-core inflect omegaconf modelscope pyarrow pyworld einops scipy regex tqdm tiktoken "x-transformers==2.11.24" "lightning==2.2.4" wget
!pip -q install -U openai-whisper

print("설치 완료")


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libao-common.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../00-libao-common_1.2.2+20180113-1.1ubuntu4_all.deb ...
Unpacking libao-common (1.2.2+20180113-1.1ubuntu4) ...
Selecting previously unselected package libao4:amd64.
Preparing to unpack .../01-libao4_1.2.2+20180113-1.1ubuntu4_amd64.deb ...
Unpacking libao4:amd64 (1.2.2+20180113-1.1ubuntu4) ...
Selecting previously unselected package libid3tag0:amd64.
Preparing to unpack .../02-libid3tag0_0.15.1b-14build1_amd64.deb ...
Unpacking libid3tag0:amd64 (0

In [2]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


## 등록 음성 업로드

In [3]:
from google.colab import files
import subprocess

def upload_wav(save_path):
    name = next(iter(files.upload()))
    subprocess.run(
        ["ffmpeg", "-y", "-i", name, "-ac", "1", "-ar", "16000", save_path],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True
    )
    return save_path

REF_TEXT = "음성 인증 실험을 위한 음성 파일입니다. 이 음성은 화자 인증과 인공지능 음성 생성 실험에 사용됩니다."
print("아래 문장을 읽어 녹음하세요.\n", REF_TEXT)

ENROLL_REF = upload_wav("/content/enroll_ref.wav")
print("등록 음성 저장 완료:", ENROLL_REF)


아래 문장을 읽어 녹음하세요.
 음성 인증 실험을 위한 음성 파일입니다. 이 음성은 화자 인증과 인공지능 음성 생성 실험에 사용됩니다.


Saving enrollment.m4a to enrollment.m4a
등록 음성 저장 완료: /content/enroll_ref.wav


In [4]:
# 등록한 음성 들어보기

from IPython.display import Audio, display

display(Audio(ENROLL_REF))

## Test 음성 업로드

In [5]:
STUDENT_ID = "20211541"  # 본인 학번으로 수정
digit = dict(zip("0123456789", "공일이삼사오육칠팔구"))
spoken_id = " ".join(digit[d] for d in STUDENT_ID)

TEST_TEXT = f"안녕하세요. 저는 숭실대학교 전자정보공학부 {spoken_id} 이요원입니다."
print("본인 정보에 맞게 수정 후 읽어 녹음하세요.\n", TEST_TEXT)

REAL_TEST = upload_wav("/content/real_test.wav")
print("시험 음성 저장 완료:", REAL_TEST)


본인 정보에 맞게 수정 후 읽어 녹음하세요.
 안녕하세요. 저는 숭실대학교 전자정보공학부 이 공 이 일 일 오 사 일 이요원입니다.


Saving test.m4a to test.m4a
시험 음성 저장 완료: /content/real_test.wav


In [6]:
# 실제 시험 음성 들어보기

display(Audio(REAL_TEST))

## TTS 음성 생성

In [7]:
import sys
from huggingface_hub import snapshot_download

COSY_MODEL = "/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B"
snapshot_download("FunAudioLLM/Fun-CosyVoice3-0.5B-2512", local_dir=COSY_MODEL)

sys.path[:0] = ["/content/CosyVoice", "/content/CosyVoice/third_party/Matcha-TTS"]
from cosyvoice.cli.cosyvoice import AutoModel

cosyvoice = AutoModel(model_dir=COSY_MODEL, fp16=torch.cuda.is_available())
print("CosyVoice3 모델 로드 완료")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

llm.rl.pt:   0%|          | 0.00/2.02G [00:00<?, ?B/s]

speech_tokenizer_v3.batch.onnx:   0%|          | 0.00/969M [00:00<?, ?B/s]

speech_tokenizer_v3.onnx:   0%|          | 0.00/969M [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
/usr/local/lib/python3.13/dist-packages/lightning/fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


CosyVoice3 모델 로드 완료


In [8]:
import torchaudio

SPOOF = "/content/spoof_cosyvoice.wav"
MODEL_PROMPT = "You are a helpful assistant.<|endofprompt|>" + REF_TEXT

results = cosyvoice.inference_zero_shot(
    TEST_TEXT, MODEL_PROMPT, ENROLL_REF, stream=False, text_frontend=False
)
speech = torch.cat([r["tts_speech"] for r in results], dim=1)

torchaudio.save(SPOOF, speech.cpu(), cosyvoice.sample_rate)
print("AI 복제 음성 생성 완료:", SPOOF)


  0%|          | 0/1 [00:00<?, ?it/s]WARNING:root:synthesis text 안녕하세요. 저는 숭실대학교 전자정보공학부 이 공 이 일 일 오 사 일 이요원입니다. too short than prompt text You are a helpful assistant.<|endofprompt|>음성 인증 실험을 위한 음성 파일입니다. 이 음성은 화자 인증과 인공지능 음성 생성 실험에 사용됩니다., this may lead to bad performance
/usr/local/lib/python3.13/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/content/CosyVoice/cosyvoice/cli/model.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with self.llm_context, torch.cuda.amp.autocast(self.fp16 is True and hasattr(self.llm, 'vllm') is False):
/content/CosyVoice/cosyvoice/cli/model.py:426: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self.fp16):
100%|██████████| 1/1 [00:13<00:00, 13.83s/it]

AI 복제 음성 생성 완료: /content/spoof_cosyvoice.wav


In [9]:
print("[ 실제 본인 음성 ]")
display(Audio(REAL_TEST))

print("[ AI 복제 음성 ]")
display(Audio(SPOOF))


[ 실제 본인 음성 ]


[ AI 복제 음성 ]


## 화자인증 모델의 인증 결과 분석

In [10]:
from speechbrain.inference.speaker import SpeakerRecognition

asv = SpeakerRecognition.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/pretrained_ecapa",
    run_opts={"device": device}
)
print("ECAPA-TDNN 모델 로드 완료")


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


hyperparams.yaml: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


...

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


label_encoder.txt: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


ECAPA-TDNN 모델 로드 완료


In [11]:
def verify(path):
    score, pred = asv.verify_files(ENROLL_REF, path)
    return float(score.squeeze()), bool(pred.squeeze())

real_score, real_prediction = verify(REAL_TEST)
spoof_score, spoof_prediction = verify(SPOOF)

for name, score, pred in [
    ("실제 본인 음성", real_score, real_prediction),
    ("AI 복제 음성", spoof_score, spoof_prediction)
]:
    print(f"[ {name} ]\nSimilarity score : {score:.4f}\nSame speaker     : {pred}\n")


[ 실제 본인 음성 ]
Similarity score : 0.7971
Same speaker     : True

[ AI 복제 음성 ]
Similarity score : 0.8673
Same speaker     : True



## CM 모델 (DF Arena 500M 딥페이크 탐지)


In [12]:
from transformers import pipeline

if "cosyvoice" in globals():
    del cosyvoice
torch.cuda.empty_cache()

cm = pipeline(
    "antispoofing",
    model="Speech-Arena-2025/DF_Arena_500M_V_1",
    trust_remote_code=True,
    device=device
)
print("DF Arena 500M 모델 로드 완료")


config.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

configuration_antispoofing.py:   0%|          | 0.00/327 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Speech-Arena-2025/DF_Arena_500M_V_1:
- configuration_antispoofing.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pipeline_antispoofing.py: 0.00B [00:00, ?B/s]

feature_extraction_antispoofing.py: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.75G [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

Device set to use cuda


DF Arena 500M 모델 로드 완료


In [13]:
import librosa

def detect_deepfake(path):
    wav, _ = librosa.load(path, sr=16000, mono=True)
    r = cm(wav)
    label = "Bona fide" if r["label"] == "bonafide" else "Spoof"
    return r["all_scores"]["bonafide"], r["all_scores"]["spoof"], label


In [14]:
cm_real_bona, cm_real_spoof, cm_real_label = detect_deepfake(REAL_TEST)
cm_ai_bona, cm_ai_spoof, cm_ai_label = detect_deepfake(SPOOF)

for name, bona, spoof, label in [
    ("실제 본인 음성", cm_real_bona, cm_real_spoof, cm_real_label),
    ("AI 복제 음성", cm_ai_bona, cm_ai_spoof, cm_ai_label)
]:
    print(f"[ {name} ]\nBona fide : {bona:.4f}\nSpoof     : {spoof:.4f}\nPrediction: {label}\n")


[ 실제 본인 음성 ]
Bona fide : 0.9797
Spoof     : 0.0203
Prediction: Bona fide

[ AI 복제 음성 ]
Bona fide : 0.0001
Spoof     : 0.9999
Prediction: Spoof



## 간단한 SASV 설계 (Cascading)

In [21]:
def cascading_sasv(asv_result, cm_result):
    return "Accept" if asv_result and cm_result == "Bona fide" else "Reject" # ASV, CM 두 모듈에서 모두 accept 되어야 최종 accpet

real_sasv = cascading_sasv(real_prediction, cm_real_label)
ai_sasv = cascading_sasv(spoof_prediction, cm_ai_label)

print("[ 실제 본인 음성 ]")
print("ASV :", "Same Speaker" if real_prediction else "Different Speaker")
print("CM  :", cm_real_label)
print("SASV:", real_sasv)

print("\n[ AI 복제 음성 ]")
print("ASV :", "Same Speaker" if spoof_prediction else "Different Speaker")
print("CM  :", cm_ai_label)
print("SASV:", ai_sasv)


[ 실제 본인 음성 ]
ASV : Same Speaker
CM  : Bona fide
SASV: Accept

[ AI 복제 음성 ]
ASV : Same Speaker
CM  : Spoof
SASV: Reject


# Part 2. SASV Score Fusion 설계

제공된 **DEV / EVAL score**를 이용해 Score Fusion을 비교한다.

CSV 형식:

` speaker, utt, key, label, asv_score, cm_score `

비교할 예시 방법은 다음 세 가지이다.

1. **Raw Score Sum**
   \[
   S = S_{ASV} + S_{CM}
   \]

2. **CM Sigmoid + Sum**
   \[
   S = S_{ASV} + \sigma(S_{CM})
   \]

3. **Weighted Sum**
   \[
   S = \alpha S_{ASV} + (1-\alpha)S_{CM}
   \]

세 방법을 **DEV에서 먼저 비교**하고, DEV SASV-EER이 가장 낮은 방법 하나를 선택한다.  
선택된 방법의 **parameter와 EER threshold를 그대로 고정**하여 EVAL에서 최종 성능을 평가한다.


## 1. DEV / EVAL score 파일 업로드


In [16]:
from google.colab import files
import pandas as pd
import numpy as np

def load_csv(message):
    print(message)
    name = next(iter(files.upload()))
    return pd.read_csv(name)

dev_df = load_csv("dev_scores.csv를 업로드하세요.")
eval_df = load_csv("eval_scores.csv를 업로드하세요.")

print("DEV :", len(dev_df), "trials")
print("EVAL:", len(eval_df), "trials")
display(dev_df.head())


dev_scores.csv를 업로드하세요.


Saving asv2019_dev_ecapa_aasist_scores.csv to asv2019_dev_ecapa_aasist_scores.csv
eval_scores.csv를 업로드하세요.


Saving asv2019_eval_ecapa_aasist_scores.csv to asv2019_eval_ecapa_aasist_scores.csv
DEV : 29548 trials
EVAL: 102579 trials


,speaker,utt,key,label,asv_score,cm_score
0,LA_0073,LA_D_4004968,bonafide,target,0.665818,6.688945
1,LA_0073,LA_D_6027798,bonafide,target,0.807858,5.577665
2,LA_0073,LA_D_3986002,bonafide,target,0.795602,4.592727
3,LA_0073,LA_D_9330492,bonafide,target,0.625112,4.411536
4,LA_0073,LA_D_1364611,bonafide,target,0.732090,4.845447


## 2. SASV-EER과 EER Threshold

`target`은 Accept, `nontarget + spoof`는 Reject 대상으로 사용한다.

DEV에서 FAR과 FRR이 가장 가까워지는 지점을 **EER threshold**로 결정한다.


In [17]:
from sklearn.metrics import roc_curve

def sasv_eer(df, score_col):
    y = (df["label"] == "target").astype(int)
    fpr, tpr, th = roc_curve(y, df[score_col])
    fnr = 1 - tpr
    i = np.argmin(np.abs(fpr - fnr))
    return (fpr[i] + fnr[i]) / 2, th[i]


## 3. DEV에서 세 가지 Score Fusion 비교


In [18]:
from scipy.special import expit

# Method 1: Raw Sum
dev_df["raw"] = dev_df["asv_score"] + dev_df["cm_score"]
eer1, th1 = sasv_eer(dev_df, "raw")

# Method 2: CM Sigmoid + Sum
dev_df["sigmoid"] = dev_df["asv_score"] + expit(dev_df["cm_score"])
eer2, th2 = sasv_eer(dev_df, "sigmoid")

# Method 3: Weighted Sum
rows = []
for a in np.arange(0, 1.01, 0.05):
    dev_df["weighted"] = a * dev_df["asv_score"] + (1-a) * dev_df["cm_score"]
    eer, th = sasv_eer(dev_df, "weighted")
    rows.append([a, eer, th])

best_w = min(rows, key=lambda x: x[1])
alpha, eer3, th3 = best_w

result = pd.DataFrame([
    ["Raw Sum", eer1, th1, None],
    ["CM Sigmoid + Sum", eer2, th2, None],
    ["Weighted Sum", eer3, th3, alpha]
], columns=["Method", "DEV EER", "Threshold", "Alpha"])

display(result.assign(**{"DEV EER (%)": result["DEV EER"]*100})
              [["Method", "DEV EER (%)", "Threshold", "Alpha"]].round(4))


,Method,DEV EER (%),Threshold,Alpha
0,Raw Sum,12.9914,5.0822,NaN
1,CM Sigmoid + Sum,0.6719,1.4132,NaN
2,Weighted Sum,1.1376,0.6652,0.95


## 4. DEV에서 가장 좋은 방법 선택

세 방법 중 **DEV SASV-EER이 가장 낮은 방법**을 최종 Score Fusion 방식으로 선택한다.


In [19]:
best = result.loc[result["DEV EER"].idxmin()]

BEST_METHOD = best["Method"]
BEST_THRESHOLD = float(best["Threshold"])
BEST_ALPHA = None if pd.isna(best["Alpha"]) else float(best["Alpha"])

print("Best method    :", BEST_METHOD)
print("DEV EER (%)    :", round(best["DEV EER"] * 100, 3))
print("Threshold      :", round(BEST_THRESHOLD, 4))
if BEST_ALPHA is not None:
    print("Best alpha     :", round(BEST_ALPHA, 2))


Best method    : CM Sigmoid + Sum
DEV EER (%)    : 0.672
Threshold      : 1.4132


## 5. 선택된 방법을 EVAL에 그대로 적용

- **ASV-EER**: Target과 Nontarget을 구분하는 화자인증 성능
- **CM-EER**: Target과 Spoof를 구분하는 스푸핑 탐지 성능
- **SASV-EER**: Target과 Nontarget + Spoof를 함께 구분하는 통합 인증 성능

EVAL에서는 DEV에서 선택한 Fusion 방식과 Weighted Sum의 α 값을 변경하지 않는다.

In [20]:
# 선택된 방법으로 EVAL fusion score 생성
if BEST_METHOD == "Raw Sum":
    eval_df["fusion"] = eval_df["asv_score"] + eval_df["cm_score"]

elif BEST_METHOD == "CM Sigmoid + Sum":
    eval_df["fusion"] = eval_df["asv_score"] + expit(eval_df["cm_score"])

else:
    eval_df["fusion"] = (
        BEST_ALPHA * eval_df["asv_score"]
        + (1-BEST_ALPHA) * eval_df["cm_score"]
    )

def get_eer(df, score_col):
    y = (df["label"] == "target").astype(int)
    fpr, tpr, _ = roc_curve(y, df[score_col])

    fnr = 1 - tpr
    i = np.argmin(abs(fpr - fnr))

    return (fpr[i] + fnr[i]) / 2

# ASV-EER: target vs nontarget
asv_df = eval_df[
    eval_df["label"].isin(["target", "nontarget"])
]

# CM-EER: target vs spoof
cm_df = eval_df[
    eval_df["label"].isin(["target", "spoof"])
]

asv_eer = get_eer(asv_df, "asv_score")
cm_eer = get_eer(cm_df, "cm_score")
sasv_eer = get_eer(eval_df, "fusion")

print("Method   :", BEST_METHOD)
print("ASV-EER  :", round(asv_eer * 100, 3), "%")
print("CM-EER   :", round(cm_eer * 100, 3), "%")
print("SASV-EER :", round(sasv_eer * 100, 3), "%")

Method   : CM Sigmoid + Sum
ASV-EER  : 0.842 %
CM-EER   : 0.671 %
SASV-EER : 0.97 %
